# Clase 195 — MLflow Tracking + Model Registry

Trackear runs (params, metrics, modelo) y promover el mejor a `Production`.

Requiere: `pip install mlflow scikit-learn`. Abrí en paralelo `mlflow ui` desde el mismo directorio.

## Setup

In [ ]:
import os, tempfile, shutil
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'mlflow_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

import mlflow
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

mlflow.set_tracking_uri(f'file:{WORK}/mlruns')
mlflow.set_experiment('housing-demo')

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print('train:', Xtr.shape, 'test:', Xte.shape)

## 1. Tracking manual — 1 run, 1 modelo

Las 3 primitivas: `log_param`, `log_metric`, `log_model`.

In [ ]:
with mlflow.start_run(run_name='lr-baseline'):
    m = LinearRegression().fit(Xtr, ytr)
    pred = m.predict(Xte)
    mlflow.log_param('model_type', 'LinearRegression')
    mlflow.log_metric('rmse_test', np.sqrt(mean_squared_error(yte, pred)))
    mlflow.log_metric('r2_test', r2_score(yte, pred))
    mlflow.sklearn.log_model(m, name='model')
    print('run_id:', mlflow.active_run().info.run_id)

## 2. Autolog — sin boilerplate

`autolog` engancha el `.fit()` y registra params + metrics + modelo automáticamente.

In [ ]:
mlflow.sklearn.autolog(log_models=True, log_datasets=False)
with mlflow.start_run(run_name='rf-autolog'):
    RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42).fit(Xtr, ytr)
    # rmse_test no se loguea solo; lo agregamos a mano si lo querés en el dashboard
mlflow.sklearn.autolog(disable=True)

## 3. Sweep — 10 runs variando hiperparámetros

In [ ]:
from itertools import product
for depth, n_est in product([3, 5, 10, 15, 20], [50, 200]):
    with mlflow.start_run(run_name=f'rf_d{depth}_n{n_est}'):
        m = RandomForestRegressor(n_estimators=n_est, max_depth=depth, n_jobs=-1, random_state=42).fit(Xtr, ytr)
        rmse = np.sqrt(mean_squared_error(yte, m.predict(Xte)))
        mlflow.log_params({'model_type': 'RandomForest', 'max_depth': depth, 'n_estimators': n_est})
        mlflow.log_metric('rmse_test', rmse)
        mlflow.log_metric('r2_test', r2_score(yte, m.predict(Xte)))
        mlflow.sklearn.log_model(m, name='model')
print('10 runs registrados.')

## 4. Búsqueda + Registry

Encontramos el mejor por `rmse_test` y lo registramos como `housing-best`.

In [ ]:
runs = mlflow.search_runs(order_by=['metrics.rmse_test ASC'], filter_string="params.model_type = 'RandomForest'")
best = runs.iloc[0]
print('mejor run:', best['run_id'], '| rmse:', round(best['metrics.rmse_test'], 4))

result = mlflow.register_model(f"runs:/{best['run_id']}/model", 'housing-best')
print('version registrada:', result.version)

In [ ]:
# Transicionar a Production (API legacy — sigue funcionando)
from mlflow.tracking import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(name='housing-best', version=result.version, stage='Production', archive_existing_versions=True)

# API moderna (MLflow 2.5+): alias
client.set_registered_model_alias('housing-best', 'champion', result.version)
print('promoted to Production + alias @champion')

## 5. Carga "en producción"

Un servicio cualquiera carga el modelo por URI lógica — no necesita saber el `run_id`.

In [ ]:
loaded = mlflow.pyfunc.load_model('models:/housing-best@champion')
preds = loaded.predict(Xte.head())
print('predicciones:', preds)
print('reales:      ', yte.head().values)

## Ejercicio guiado

1. Agregá un run con `XGBRegressor` (`pip install xgboost`) y compará vs el mejor RF.
2. Usá `mlflow.search_runs` con filtro `"metrics.r2_test > 0.7"` y ordená por `rmse_test`.
3. Promové un challenger con alias `@challenger` y escribí una celda que cargue ambos (`@champion`, `@challenger`) y compare predicciones sobre las primeras 100 filas de test.

## Conclusiones

- `log_param`/`log_metric`/`log_model` son la API mínima viable; **autolog** quita boilerplate cuando el framework lo soporta.
- El **Model Registry** desacopla `entrenar` de `servir`: la API en producción referencia `models:/housing-best@champion`, no un `run_id`.
- Promover de `champion` a otro modelo es **una llamada API**, no un redeploy del servidor.